# 🧠 Gözleri Açılan Sinek (Duyu Motorlu FlyOpt vs PSO)

Bu testte sineğe mutlak GPS (X,Y) koordinatları yerine, bir canlının doğada yapacağı gibi **Koku Gradyanı (Hedefin Yönü)** veriliyor. Bakalım sinek kokuyu alınca PSO'yu geçebilecek mi?

In [ ]:
# 1. GEREKSİNİMLER VE DRIVE BAĞLANTISI
from google.colab import drive
import sys
import os
import torch
import numpy as np
import pandas as pd
import time
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/fly_op'
if not os.path.exists(project_path):
    project_path = '/content/drive/MyDrive/fly_op/fly_op'

sys.path.append(project_path)
sys.path.append(os.path.join(project_path, 'src'))
os.chdir(project_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Aktif Donanım: {device}')

In [ ]:
# 2. ALGORİTMALARIN TANIMLANMASI (DUYUSAL SİNEK GÜNCELLEMESİ)
from flyopt.variants.rate_brain import RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode

def sphere(x): return torch.sum(x**2, dim=1)
def rastrigin(x): return 10 * x.shape[1] + torch.sum(x**2 - 10 * torch.cos(2 * np.pi * x), dim=1)

# KLASİK PSO
def pso_optimize(func, dim=2, num_particles=1000, iters=100):
    x = (torch.rand((num_particles, dim), device=device) * 10 - 5)
    v = torch.zeros_like(x)
    pbest = x.clone()
    pbest_obj = func(x)
    gbest = pbest[torch.argmin(pbest_obj)].clone()
    gbest_obj = torch.min(pbest_obj)
    w, c1, c2 = 0.5, 1.5, 1.5
    history = []
    start = time.time()
    for _ in range(iters):
        r1, r2 = torch.rand((num_particles, dim), device=device), torch.rand((num_particles, dim), device=device)
        v = w * v + c1 * r1 * (pbest - x) + c2 * r2 * (gbest - x)
        x = x + v
        obj = func(x)
        mask = obj < pbest_obj
        pbest[mask] = x[mask]
        pbest_obj[mask] = obj[mask]
        if torch.min(pbest_obj) < gbest_obj:
            gbest_obj = torch.min(pbest_obj)
        history.append(gbest_obj.item())
    return history, time.time() - start

# GÖZLERİ AÇILAN BİYOLOJİK MOTOR (SENSORY FLYOPT)
def fly_optimize_sensory(func, brain, dim=2, num_particles=1000, iters=100):
    # Sinekler uzaya bırakılır ve duyuları (grad) açılır
    x = (torch.rand((num_particles, dim), device=device) * 10 - 5)
    x.requires_grad_(True)
    
    best_obj = func(x).detach()
    history = []
    input_pad = torch.zeros((num_particles, 4 - dim), device=device)
    
    start = time.time()
    for _ in range(iters):
        # 1. Koku (Gradient) Algılayıcılar
        obj = func(x)
        obj.sum().backward()
        
        with torch.no_grad():
            grad = x.grad.clone()
            x.grad.zero_()
            
            # Kokunun yönü (gradientin tersi hedeftir)
            grad_norm = torch.nn.functional.normalize(grad, p=2, dim=1)
            smell_input = -grad_norm  # "Yemek şu yönde"
            
            # Sineğe mutlak GPS değil, koku yönünü veriyoruz!
            env_input = torch.cat([smell_input, input_pad], dim=1)
            fly_step = brain(env_input)
            
            # Sinek kokuyu 20 adım işleyip adım atıyor
            move_vec = fly_step[:, :dim] * 0.1
            x_new = x.detach() + move_vec
            
            # Hayatta kalma kontrolü
            new_obj = func(x_new)
            mask = new_obj < best_obj
            
            # Temsili x güncellemesi (grad kopmasın diye özenle)
            x_updated = x.detach().clone()
            x_updated[mask] = x_new[mask]
            x = x_updated.clone()
            x.requires_grad_(True)
            
            best_obj[mask] = new_obj[mask]
            history.append(torch.min(best_obj).item())
            
    return history, time.time() - start

In [ ]:
# 3. ERKEK SİNEK (Male CNS) YÜKLEMESİ
import glob

# Drive yapısındaki data klasörünü otomatik bul
found_files = glob.glob('/content/drive/MyDrive/**/malecns_adjacency.npz', recursive=True)
if found_files:
    DATA_PROCESSED = os.path.dirname(found_files[0])
    print(f"Veri bulundu: {DATA_PROCESSED}")
else:
    DATA_PROCESSED = f"{project_path}/data/processed"

print("Ağ yükleniyor...")
base_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
afferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy")
efferent = np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

encode_full, decode_full = select_connected_encode_decode(base_weights, afferent, efferent, n_encode=4, n_decode=4, max_hops=6, n_encode_candidates=200, seed=9000)
sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, 3000, seed=9000)

cfg = RateBrainConfig(dim=2, n_readout=len(decode_idx), T=20, decode_scale=0.5, train_gain=True)
brain = RateBrain(sub_real, encode_idx, decode_idx, cfg, seed=42).to(device)
print("Duyusal Biyolojik Motor (T=20) GPU'ya yüklendi!")

In [ ]:
# 4. KAFES DÖVÜŞÜ (GÖZLERİ AÇILMIŞ SİNEK VS PSO)
NUM_AGENTS = 2000
ITERS = 100

print(f"1. RAUNT: SPHERE (Kolay Görev - {NUM_AGENTS} Ajan)")
pso_s, time_pso_s = pso_optimize(sphere, num_particles=NUM_AGENTS, iters=ITERS)
fly_s, time_fly_s = fly_optimize_sensory(sphere, brain, num_particles=NUM_AGENTS, iters=ITERS)

print(f"\n2. RAUNT: RASTRIGIN (Zorlu/Tuzaklı Görev - {NUM_AGENTS} Ajan)")
pso_r, time_pso_r = pso_optimize(rastrigin, num_particles=NUM_AGENTS, iters=ITERS)
fly_r, time_fly_r = fly_optimize_sensory(rastrigin, brain, num_particles=NUM_AGENTS, iters=ITERS)

# --- GRAFİKLER VE KAYIT --- 
sns.set_theme(style="darkgrid")
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(pso_s, label=f'PSO ({time_pso_s:.1f}s)', color='red', linewidth=2)
plt.plot(fly_s, label=f'Duyusal FlyOpt ({time_fly_s:.1f}s)', color='blue', linewidth=2)
plt.title('Sphere Optimizasyonu (Koku-Bazlı Sinek)')
plt.xlabel('İterasyon')
plt.ylabel('Hata Payı (Log)')
plt.yscale('log')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(pso_r, label=f'PSO ({time_pso_r:.1f}s)', color='red', linewidth=2)
plt.plot(fly_r, label=f'Duyusal FlyOpt ({time_fly_r:.1f}s)', color='blue', linewidth=2)
plt.title('Rastrigin Optimizasyonu (Koku-Bazlı Sinek)')
plt.xlabel('İterasyon')
plt.ylabel('Hata Payı (Log)')
plt.yscale('log')
plt.legend()

plt.tight_layout()
grafik_adi = 'Duyusal_Sinek_vs_PSO.png'
plt.savefig(grafik_adi, dpi=300)
plt.show()

df_sonuclar = pd.DataFrame({
    "Iterasyon": list(range(1, ITERS+1)),
    "PSO_Sphere": pso_s, "Fly_Sphere": fly_s,
    "PSO_Rastrigin": pso_r, "Fly_Rastrigin": fly_r
})
csv_adi = 'Duyusal_Sinek_Verileri.csv'
df_sonuclar.to_csv(csv_adi, index=False)

print("\n✅ İşlem bitti! Yeni sonuçlar bilgisayarınıza indiriliyor...")
try:
    files.download(grafik_adi)
    files.download(csv_adi)
except:
    print("İndirme engellendi, sol panelden manuel indirebilirsiniz.")